# Population Fidelity on better-sampled cells

A survey cell is a small sample. The benchmark merges rare profiles until a cell
holds 20 respondents, and because some respondents skip an item, the distribution
a cell contributes to one question rests on 4 to 114 valid answers, 31 to 35 at
the median. The survey's pairwise distances therefore carry sampling noise, and a
structure score near zero could in principle come from that noise rather than
from the model. This notebook scores every run again on the cells with at least
$t$ valid survey answers to the question,

$$I_t = \{\, i \in I : n_i \ge t \,\}, \qquad t \in \{20, 30, 40, 50, 75, 100\},$$

beside the whole set of retained cells, $t = 0$. The count $n_i$ is the weight of
the respondent-weighted readings, `respondent_counts`: valid survey answers, not
the model's. Every score is `group_fidelity` on the kept cells, so nothing about
the metric changes, only the cells it reads. A set with fewer than three cells is
left unscored. Paired comparisons keep the cells common to every run of the tuned
and the as-released condition, then apply the same minimum.

Two simulators that are right by construction say what the survey's own sampling
noise leaves room for. Both start from the survey's cell distributions and redraw
each cell's answers with replacement, at the cell's own size. `exact` scores those
distributions themselves against a redrawn survey: a simulator that returns every
cell's distribution exactly, as next-token probabilities would. `sampled` scores a
second redrawn copy against the first: a simulator that also answers once per
respondent, as full answers do. Their scores are averaged over the redraws.

Reads the runs the manuscript reports, every replicate, and reconciles the rows at
$t = 0$ against `population_fidelity_groups.csv` and `population_fidelity_paired.csv`.
Writes `population_fidelity_cell_size.csv`, `population_fidelity_cell_size_paired.csv`,
`population_fidelity_cell_size_survey.csv`, `population_fidelity_cell_size_summary.csv`
and `figures/fidelity/cell_size/fig_fidelity_cell_size_structure`.

In [1]:
from culture.fidelity import GROUPS_TABLE, PAIRED_TABLE
from culture.fidelity_baseline import manuscript_runs
from culture.fidelity_cell_size import (
    CELL_SIZE_PAIRED_TABLE,
    CELL_SIZE_SUMMARY_TABLE,
    CELL_SIZE_SURVEY_TABLE,
    CELL_SIZE_TABLE,
    EVERY_QUESTION,
    build_cell_size,
    cell_size_plate,
    cell_size_summary,
    check_cell_size,
    survey_reference,
)
from culture.tables import read_csv
from machine_bias_reproduction.io_utils import write_csv

## Build

In [2]:
runs = manuscript_runs(every_replicate=True)
frame, paired = build_cell_size(runs=runs)
print(len(frame), "rows and", len(paired), "paired rows over", frame["source"].nunique(), "runs and", frame["min_respondents"].nunique(), "minimums")

2156 rows and 1344 paired rows over 57 runs and 7 minimums


## Reconciling against the fidelity tables

The rows at $t = 0$ are the pooled rows of the fidelity tables. The check asserts
that they equal `population_fidelity_groups.csv` and
`population_fidelity_paired.csv` on every column, that a larger minimum never
keeps more cells, and that a set is scored exactly when it holds three cells.

In [3]:
groups, published_paired = read_csv(GROUPS_TABLE), read_csv(PAIRED_TABLE)
assert groups is not None and published_paired is not None
matched = check_cell_size(frame, paired, groups, published_paired)
print(matched, "rows carry the published scores unchanged")

500 rows carry the published scores unchanged


## What sampling noise leaves room for

One row per question, minimum and simulator: the mean of each score over 500
redraws and the 2.5th and 97.5th percentiles across them.

In [4]:
survey = survey_reference()
print(len(survey), "rows over", survey["question"].nunique(), "questions and", int(survey["n_resamples"].max()), "redraws")

56 rows over 4 questions and 500 redraws


## The tables

`population_fidelity_cell_size.csv` and `population_fidelity_cell_size_paired.csv`
hold the pooled row of every run at every minimum, with `n_respondents`, the valid
survey answers behind the kept cells. `population_fidelity_cell_size_survey.csv`
holds the two simulators. `population_fidelity_cell_size_summary.csv` reads the
first run: one row per minimum and question, closed by an `all` row over the 164
pooled combinations, with the medians, the count of combinations whose lowest
component is structure, the zero scores, the range of the simulators' structure
and PFS across questions, and the mean paired change of each cultural target.

In [5]:
summary = cell_size_summary(frame, paired, survey)
write_csv(frame, CELL_SIZE_TABLE)
write_csv(paired, CELL_SIZE_PAIRED_TABLE)
write_csv(survey, CELL_SIZE_SURVEY_TABLE)
write_csv(summary, CELL_SIZE_SUMMARY_TABLE)
print(len(summary), "summary rows over", summary["min_respondents"].nunique(), "minimums")

35 summary rows over 7 minimums


## The plate

One panel per question. The boxes span the structure of the 41 combinations of
the first run at each minimum, the lines the two simulators with the 2.5th to
97.5th percentile of their redraws. A minimum is drawn when every combination
can be scored at it, which leaves out 100.

In [6]:
files = cell_size_plate(frame, survey)
print(len(files), "files")

2 files


## Reading the table

Structure stays the lowest component in at least 157 of the 164 combinations up
to a minimum of 50, where a third of the valid answers remain, and in 153 at 75,
where 10 to 20 cells remain. Its median stays between 0.031 and 0.044 up to 50.
Over the same range the structure an exactly right simulator would score rises
from 0.72 to 0.85 across questions to 0.79 to 0.90, and that of a sampled one
from 0.55 to 0.74 to 0.65 to 0.82. Keeping the better-sampled cells makes the
survey's structure easier to recover and leaves the models where they were.

A minimum of 100 cannot be scored: one to four cells hold that many answers, a
single one on political ideology.

The mean paired change of the Mexican target stays negative at every minimum. That
of the German target stays negative up to 40 and is close to zero at 50 and 75.

In [7]:
pooled = summary[summary["question"].eq(EVERY_QUESTION)].set_index("min_respondents")
levels = ["n_scored", "n_cells_min", "n_cells_max", "share_respondents_mean", "score_structure_median", "pfs_median", "n_structure_binding", "n_zero_pfs"]
print(pooled[levels].round(3).to_string())
simulators = ["exact_score_structure_min", "exact_score_structure_max", "sampled_score_structure_min", "sampled_score_structure_max"]
changes = ["german_delta_pfs_mean", "german_n_pfs_up", "german_n_pfs_down", "spanish_mx_delta_pfs_mean", "spanish_mx_n_pfs_up", "spanish_mx_n_pfs_down"]
print(pooled[simulators + changes].round(3).to_string())

                 n_scored  n_cells_min  n_cells_max  share_respondents_mean  score_structure_median  pfs_median  n_structure_binding  n_zero_pfs
min_respondents                                                                                                                                 
0                     164          606          687                   1.000                   0.042       0.220                  160          49
20                    164          543          679                   0.976                   0.039       0.223                  160          51
30                    164          340          448                   0.757                   0.034       0.205                  161          55
40                    164          179          272                   0.521                   0.031       0.216                  160          52
50                    164           82          155                   0.326                   0.044       0.231                  1